# 春第 2 回｜表データと条件抽出

2024 年 1 月 14 日に，ニューヨーク周辺から朝出る予定だった便を調べます。本文の BOS 行きから，シカゴ・オヘア（ORD）またはアトランタ（ATL）へ行先を変えます。8 時台の表を残し，調べる時間を広げた表と比べてください。

共通課題は 1〜3 の約 20 分です。目標は件数を出すだけでなく，**どの条件を変えると，どんな便が新しく見つかったかを説明すること**です。末尾の到着記録の課題は任意です。

## 準備

Colab では「ドライブにコピーを保存」で自分用のノートを作り，上から準備セルを実行します。CSV は公開 URL から読み込むため，アップロードは不要です。ローカルでは，実習 ZIP を展開してノートブックと CSV を同じフォルダーに置くと，手元のファイルを使えます。

最初のセルはファイルの置き場所に対応する準備です。そのまま実行してください。出力は予定便の総数 `32347` です。この表は当時の記録であり，現在の時刻表ではありません。

In [ ]:
from pathlib import Path

import pandas as pd

file_name = "nyc-flights-2024-01.csv"
csv_path = Path("data") / file_name
if not csv_path.is_file():
    csv_path = Path("../data") / file_name
if not csv_path.is_file():
    csv_path = Path(file_name)
if not csv_path.is_file():
    csv_path = "https://raw.githubusercontent.com/sohei-takagi/python-ds-course-materials/main/data/" + file_name

flights = pd.read_csv(csv_path)
len(flights)

## 1. 行先と基準の表（目安 4 分）

ORD または ATL を選びます。下の `target_destination` に選んだコードを指定してください。日付は 1 月 14 日のままにします。

先に行先を選び，そこから日付を選びます。この時点では時間を絞りません。あとで時間帯を広げるために，その日の全時刻を含む `day_flights` を残しておきます。

In [ ]:
target_destination = "ORD"
selected_day = "2024-01-14"

is_destination = flights["destination"] == target_destination
route_flights = flights.loc[is_destination]
on_day = route_flights["date"] == selected_day
day_flights = route_flights.loc[on_day]

columns = ["date", "origin", "destination", "carrier", "scheduled_departure"]
day_flights[columns].head()

次に 8:00 以上・9:00 未満の予定便を選びます。`departure` は **その日の全時刻** の出発予定時刻の列です。`baseline` は，比較の基準として残す 8 時台の表です。

表示された便の出発空港と予定時刻を読んでください。`800` は 8:00 です。便が表示される順番は，必ずしも時刻順ではありません。

`carrier` は航空会社のコードです。同じ空港・予定時刻でも，航空会社が異なる行があります。時刻だけで同じ便と判断せず，元の行を一行ずつ数えます。

In [ ]:
departure = day_flights["scheduled_departure"]
in_eight = (departure >= 800) & (departure < 900)
baseline = day_flights.loc[in_eight]

print(len(baseline))
baseline[columns]

**ここに記録します。**

- 選んだ行先：
- 8 時台の予定便数：
- 表にあった出発空港と時刻：

## 2. 時間帯を広げる（目安 10 分）

次のどちらかを選びます。先に，どの時刻の便が新しく含まれるはずかを言葉で書いてください。

- **7 時台も含める：** 下限を `700` に変え，上限は `900` のままにする。
- **9 時台も含める：** 下限は `800` のままにして，上限を `1000` に変える。

どちらを選んでも，行先と日付はそのままです。一度に両方の境界を変えず，一つの変更と増えた便を対応させます。

**自分が調べる時間帯と，増えると予想した記録：**

（ここに書く）

In [ ]:
start_time = 800
end_time = 900

# 上の二つの値のうち，選んだ境界を一つだけ変更する。
at_or_after_start = departure >= start_time

下のセルの最初の行で，`before_end =` の右側に「出発予定時刻が上限より前」という比較を書いてください。本文の「時間帯を選ぶ」が手掛かりになります。

次に，**下の 5 行すべての先頭にある `#` を外して**実行します。`#` があるままだとコメントなので，計算は行われません。2 行目以降は書き写す必要はありません。入力した比較がどこに使われるかを読みながら実行してください。

`expanded` は変更後の表です。`baseline` は上書きせずに残します。

In [ ]:
# before_end =
# in_window = at_or_after_start & before_end
# expanded = day_flights.loc[in_window]
# print(len(expanded))
# expanded[columns]

<details>
<summary>比較式を確認したいとき</summary>

`departure` の値が `end_time` より小さいかを比べます。

```python
before_end = departure < end_time
```

上限を含めないので `<` を使います。たとえば上限が `1000` なら 9:59 は入りますが，10:00 は入りません。

`NameError` が出た場合は，`before_end` の行の `#` を外したか，その前の準備・行先・時間帯のセルを実行したかを確かめます。下限や上限を変更した後は，比較を作るセルと抽出するセルをどちらも再実行してください。

</details>

上の変更後の表と，次の基準の表を比べます。新しく見つかった便の予定時刻が，自分で加えた時間帯に入るかを読んでください。

In [ ]:
print(len(baseline))
baseline[columns]

**比較の記録**

- 変更前の条件と件数：
- 変更後の条件と件数：
- 新しく含まれた便の出発空港と予定時刻：

<details>
<summary>結果の件数を照合する</summary>

1 月 14 日の記録について，次の件数になります。件数だけが同じでも同じ便を選んだとは限らないので，出発空港と予定時刻も読んでください。

| 行先 | 8:00 以上・9:00 未満 | 7:00 以上・9:00 未満 | 8:00 以上・10:00 未満 |
|---|---:|---:|---:|
| ORD | 2 | 5 | 5 |
| ATL | 2 | 6 | 5 |

基準の件数のままの場合は，下限か上限を変えた後に，判定と抽出も実行し直したかを確かめます。時間を広げるときは，`baseline` の中だけを探すのではなく，全時刻を持つ `day_flights` から選び直します。

</details>

## 3. 見つかったことを伝える（目安 6 分）

自分が選んだ時間帯について，次の三点を短い段落にまとめてください。表の件数を読み上げるだけでなく，条件の変更と実際の記録を結び付けます。

1. いつの，どこへ向かう予定便を調べ，時間のどちらの境界を変えたか。
2. 何便から何便になり，どの空港・時刻の便が新しく見つかったか。
3. 変更前の `baseline` だけから選び直しても，新しい時間帯の便が見つからないのはなぜか。

**説明を書く欄：**

（ここに書く）

今回の表から，その便が実際にその時刻に出発したかも分かるでしょうか。`scheduled_departure` が何を記録した列かを踏まえ，最後に一文添えてください。

提出前にノートを保存し，最初から実行して，自分の条件と説明に合った結果になるかを確かめます。提出するのは，選んだ条件・実行結果・上の説明が入ったノートブックです。

## 発展：到着記録も調べる（任意）

時間に余裕があれば，自分が作った `expanded` について，次を調べてください。共通課題の提出に必要な追加作業ではありません。

- 到着の遅れが 15 分未満，15 分以上，空欄の便はそれぞれ何便か。
- 三つの件数を足すと `len(expanded)` と一致するか。
- 「15 分以上」と「15 分未満ではなかった」を同じ件数だと考えてよいか。今回一致していても，いつも一致すると言えるか。

本文の「予定と到着の記録」を参考に，必要なコードを下のセルへ書きます。

In [ ]:
# 任意：expanded の arrival_delay を取り出して調べる。